In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

# Módulos de sklearn
from sklearn.neighbors import KNeighborsRegressor   # Algoritmo que vamos a entrenar, regresion
from sklearn.model_selection import cross_val_score, KFold, train_test_split, GridSearchCV
from sklearn.metrics import classification_report  # Métricas

In [2]:
datos=pd.read_csv("hitters.csv")

In [3]:
# Objetivo: Predecir el salario de los besibolistas

In [4]:
datos.head()

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,League,Division,PutOuts,Assists,Errors,Salary,NewLeague
0,293,66,1,30,29,14,1,293,66,1,30,29,14,A,E,446,33,20,NaN,A
1,315,81,7,24,38,39,14,3449,835,69,321,414,375,N,W,632,43,10,475.0,N
2,479,130,18,66,72,76,3,1624,457,63,224,266,263,A,W,880,82,14,480.0,A
3,496,141,20,65,78,37,11,5628,1575,225,828,838,354,N,E,200,11,3,500.0,N
4,321,87,10,39,42,30,2,396,101,12,48,46,33,N,E,805,40,4,91.5,N


In [5]:
datos.columns

Index(['AtBat', 'Hits', 'HmRun', 'Runs', 'RBI', 'Walks', 'Years', 'CAtBat',
       'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks', 'League', 'Division',
       'PutOuts', 'Assists', 'Errors', 'Salary', 'NewLeague'],
      dtype='object')

In [6]:
# Atributos
X=datos[['AtBat', 'Hits', 'HmRun', 'Runs', 'RBI', 'Walks', 'Years',
        'League', 'Division']]
# target
y=datos['Salary']

In [7]:
X.head()

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,League,Division
0,293,66,1,30,29,14,1,A,E
1,315,81,7,24,38,39,14,N,W
2,479,130,18,66,72,76,3,A,W
3,496,141,20,65,78,37,11,N,E
4,321,87,10,39,42,30,2,N,E


In [8]:
#Análisis descriptivo de los datos
X.describe()  # Solo toma las numéricas

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years
count,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000
mean,380.928571,101.024845,10.770186,50.909938,48.027950,38.742236,7.444099
std,153.404981,46.454741,8.709037,26.024095,26.166895,21.639327,4.926087
min,16.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,255.250000,64.000000,4.000000,30.250000,28.000000,22.000000,4.000000
50%,379.500000,96.000000,8.000000,48.000000,44.000000,35.000000,6.000000
75%,512.000000,137.000000,16.000000,69.000000,64.750000,53.000000,11.000000
max,687.000000,238.000000,40.000000,130.000000,121.000000,105.000000,24.000000


In [9]:
# Tenemos los dato de 322 bateadores
X.shape

(322, 9)

In [10]:
# IMPORTANTE: Siempre que se considere un algoritmo que tome en cuenta la 
# distancia para hacer predicciones, hay que eliminar las unidades
# de medición estandarizado los valores de las variables numéricas

In [11]:
X.info()
# No observamos datos faltantes en los atributos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   AtBat     322 non-null    int64 
 1   Hits      322 non-null    int64 
 2   HmRun     322 non-null    int64 
 3   Runs      322 non-null    int64 
 4   RBI       322 non-null    int64 
 5   Walks     322 non-null    int64 
 6   Years     322 non-null    int64 
 7   League    322 non-null    object
 8   Division  322 non-null    object
dtypes: int64(7), object(2)
memory usage: 22.8+ KB


In [12]:
y.describe()

count     263.000000
mean      535.925882
std       451.118681
min        67.500000
25%       190.000000
50%       425.000000
75%       750.000000
max      2460.000000
Name: Salary, dtype: float64

In [13]:
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 322 entries, 0 to 321
Series name: Salary
Non-Null Count  Dtype  
--------------  -----  
263 non-null    float64
dtypes: float64(1)
memory usage: 2.6 KB


In [14]:
# Tenemos datos nulos: 59
y.isnull().sum()

59

In [15]:
# Estas 59 observaciones del target se tiene que elimiar. 

In [16]:
datos=datos[y.isnull()==False] # Eliminamos los dato nulos

In [17]:
# Con los datos completos.

# Atributos
X=datos[['AtBat', 'Hits', 'HmRun', 'Runs', 'RBI', 'Walks', 'Years',
        'League', 'Division']]
# target
y=datos['Salary']

In [18]:
y.info()

<class 'pandas.core.series.Series'>
Int64Index: 263 entries, 1 to 321
Series name: Salary
Non-Null Count  Dtype  
--------------  -----  
263 non-null    float64
dtypes: float64(1)
memory usage: 4.1 KB


In [19]:
y.isnull().sum()  # Observamos que ya no tenemos datos faltanes.

0

In [20]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 263 entries, 1 to 321
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   AtBat     263 non-null    int64 
 1   Hits      263 non-null    int64 
 2   HmRun     263 non-null    int64 
 3   Runs      263 non-null    int64 
 4   RBI       263 non-null    int64 
 5   Walks     263 non-null    int64 
 6   Years     263 non-null    int64 
 7   League    263 non-null    object
 8   Division  263 non-null    object
dtypes: int64(7), object(2)
memory usage: 20.5+ KB


In [21]:
X["League"].value_counts()

A    139
N    124
Name: League, dtype: int64

In [22]:
X.Division.value_counts()

W    134
E    129
Name: Division, dtype: int64

In [23]:
# Para manejar las variables categóricas CUALITATIVAS.
X_dummies=pd.get_dummies(X, columns=["League","Division"])
X_dummies   # <- Estos son los atributos con los que vamos a trabajar




,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,League_A,League_N,Division_E,Division_W
1,315,81,7,24,38,39,14,0,1,0,1
2,479,130,18,66,72,76,3,1,0,0,1
3,496,141,20,65,78,37,11,0,1,1,0
4,321,87,10,39,42,30,2,0,1,1,0
5,594,169,4,74,51,35,11,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
317,497,127,7,65,48,37,5,0,1,1,0
318,492,136,5,76,50,94,12,1,0,1,0
319,475,126,3,61,43,52,6,1,0,0,1
320,573,144,9,85,60,78,8,1,0,1,0


In [24]:
#Para estandarizar los datos / eliminar las unidades de medición.
# z.score o tipificación de los datos o estandarización
#   Z= (X- X.mean())/X.std()     -> [-3 , 3]

# Min-Max
#   T = (X -X.min())/(X.max()-X.min())   ->  [0, 1] 

# Solo se reescalan las variables numéricas/continuas u ordinales
# NUNCA las dummies

In [25]:
#Vamos a aplicar una transformación MinMax
from sklearn.preprocessing import MinMaxScaler , StandardScaler

# No hay que perder de vista que los conjuntos de entrenamiento y prueba
# tienen que ser independientes... 
# Si reescalamos los datos y luego los dividimos en entrenamiento y prueba
# vamos a tener un problema de "FUGA" de información. 

# Lo correcto es primero divir el conjunto de datos en entrenamiento y prueba
# y luego reescalarlos...¿? Cómo es la estrategia.

# Paso 1. Divir el conjunto de datos
X_train, X_test, y_train, y_test = train_test_split(X_dummies,y, 
                                                    test_size=0.20,
                                                    random_state=1234)
# Paso 2. Defino de la estrategia de escalamiento.
escala=MinMaxScaler()  

# Paso 3. Aprender la estrategia de escalamiento
escala.fit(X_train)

# Paso 3.1 Con la estrategia aprendida, reescalo los datos
X_train_std=escala.transform(X_train)   

# Los pasos 3 y 3.1. los pueden escribir como
# X_std=escala.fit_transform(X_train)

#Paso 4. Con los datos escalados entrenamos el algoritmo.
modelo=KNeighborsRegressor(n_neighbors=5)
modelo.fit(X_train_std,y_train)

KNeighborsRegressor()

In [26]:
# Vemos el deasempeño predictivo
# Paso 5. Hacer las predicciones, pero para hacer las predicciones
# necesitamos los datos de prueba reescalados
# Los vamos a reescalar utilizando la estrategia de reescalamiento
# que aprendimos de los datos de entrenamiento
X_test_std=escala.transform(X_test)  # Solo transformamos...
y_pred=modelo.predict(X_test_std)
y_pred

array([ 112.3   ,  520.3334,  462.    ,  309.    ,  317.1666,  817.7332,
        881.1666,  558.5   ,  767.6666,  425.    ,  670.    ,  633.    ,
        828.    ,  716.    ,  508.6666,  165.5   ,  371.3334,  839.7332,
        220.    , 1057.5   ,  184.    ,  270.    ,  154.6   ,  359.    ,
       1050.1666,  801.    ,  241.3   ,  474.    ,  550.4   ,  344.3   ,
        537.1666,  177.3334,  245.8   ,  118.3   ,  109.3   ,  295.    ,
        988.0666,  235.    ,  102.3   ,  740.    ,  405.3334,  451.    ,
        560.3334,  665.    ,  235.5   ,  291.5   ,  128.6   ,  483.    ,
        583.9   ,  465.8334,  171.5   ,  756.    ,  809.    ])

In [27]:
# La metrica que ocupamos es la raiz cuadrada del error cuadratico medio
np.sqrt((y_test-y_pred)**2).mean()

230.2273547169811

In [28]:
# Error depende del conjunto de prueba y entrenamiento y del modelo.

In [29]:
# Optimizar el hiperparámetro. / Calibrar el modelo

from sklearn.pipeline import make_pipeline

# paso 1. Definir los posibles valores de los hiperparámetros
espacio_parametros={"kneighborsregressor__n_neighbors":np.arange(3,30)}

# Paso 2. Secuencia para entrenar el modelo
# como existe un preprocesamiento (reescalar los datos)
modelo= make_pipeline(MinMaxScaler(),KNeighborsRegressor())

# Paso 3. Definir los modelos que vamos a comparar
modelos_candidatos=GridSearchCV(modelo,param_grid=espacio_parametros,
                               scoring="neg_root_mean_squared_error",
                               cv=10,n_jobs=-1)

# Paso 4. Entrenar los modelos candidatos
modelos_candidatos.fit(X_dummies,y)

# Imprimimos los resultados:
print("La mejor calibración es:", modelos_candidatos.best_params_)
print("La mejor métrica es:", -modelos_candidatos.best_score_)

La mejor calibración es: {'kneighborsregressor__n_neighbors': 15}
La mejor métrica es: 351.8421081554632


In [30]:
# Optimizar el hiperparámetro. / Calibrar el modelo
# Utilizamos un escalamiento StandarScaler


from sklearn.pipeline import make_pipeline

# paso 1. Definir los posibles valores de los hiperparámetros
espacio_parametros={"kneighborsregressor__n_neighbors":np.arange(3,30)}

# Paso 2. Secuencia para entrenar el modelo
# como existe un preprocesamiento (reescalar los datos)
modelo= make_pipeline(StandardScaler(),KNeighborsRegressor())

# Paso 3. Definir los modelos que vamos a comparar
modelos_candidatos=GridSearchCV(modelo,param_grid=espacio_parametros,
                               scoring="neg_root_mean_squared_error",
                               cv=10,n_jobs=-1)

# Paso 4. Entrenar los modelos candidatos
modelos_candidatos.fit(X_dummies,y)

# Imprimimos los resultados:
print("La mejor calibración es:", modelos_candidatos.best_params_)
print("La mejor métrica es:", -modelos_candidatos.best_score_)

La mejor calibración es: {'kneighborsregressor__n_neighbors': 13}
La mejor métrica es: 348.2276790142503


In [31]:
# Con estos valores ya tengo un modelo final.
# Tengo que entrenar el modelo utiliza TODOS los datos...
# 1. ¿Cual modelo?
modelo=KNeighborsRegressor(n_neighbors=13)
# 2. TODOS los datos pero estandarizados...
escala=StandardScaler()
X_std=escala.fit_transform(X_dummies)
# 3. Entrenamos el modelo
modelo.fit(X_std,y)
# 4. Este modelo ya me sirve para haceer predicciones...

KNeighborsRegressor(n_neighbors=13)

In [32]:
X_dummies.columns

Index(['AtBat', 'Hits', 'HmRun', 'Runs', 'RBI', 'Walks', 'Years', 'League_A',
       'League_N', 'Division_E', 'Division_W'],
      dtype='object')

In [33]:
X_nuevo=pd.DataFrame({'AtBat':[350], 'Hits':[200], 'HmRun':[5], 
                      'Runs':[25], 'RBI':[18], 'Walks':[5], 'Years':[3],
                      'League_A':[1],'League_N':[0], 'Division_E':[0],
                      'Division_W':[1]})



X_nuevo

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,League_A,League_N,Division_E,Division_W
0,350,200,5,25,18,5,3,1,0,0,1


In [34]:
# ¿Cuál sería el salario de este jugador?
# Estandarizamos primero los datos: Uitlizando la estrategia
# que aprendimos cuando estandarizamos TODOS los datos
X_nuevo_std=escala.transform(X_nuevo)
X_nuevo_std

array([[-0.36484881,  2.04645246, -0.75737255, -1.1668823 , -1.29625265,
        -1.66602953, -0.90120024,  0.94450322, -0.94450322, -0.98116592,
         0.98116592]])

In [35]:
modelo.predict(X_nuevo_std)

array([206.65384615])


El salario del jugador es:   
206.65 +/- 348.22


In [36]:
# 1.- Vamos a corregir la estandarizamos de las dummies.
# 2.- K vecinos: ponderamos...  
# 3.- La métrica euclideana es la mejor para este problema?
# 4.- Inteligencia colectiva... 


In [37]:
# Estandarizamos solo las variables numéricas y no las dummies.
# Es decir, las variables: AtBat	Hits	HmRun	Runs	RBI	Walks	Years
from sklearn.compose import make_column_transformer   # Operaciones por columnas
from sklearn.preprocessing import OneHotEncoder       # equivalente al Get dummies

column_transf=make_column_transformer(
(StandardScaler(),["AtBat","Hits","HmRun","Runs","RBI","Walks","Years"]),
(OneHotEncoder(),["League","Division"]),  # Este renglon puede sobrar si antes se hacen las dummies
remainder="passthrough")

In [38]:
# Optimizar el hiperparámetro. / Calibrar el modelo
# Utilizamos un escalamiento StandarScaler

# paso 1. Definir los posibles valores de los hiperparámetros
espacio_parametros={"kneighborsregressor__n_neighbors":np.arange(3,30)}

# Paso 2. Secuencia para entrenar el modelo
# como existe un preprocesamiento (reescalar los datos)
modelo= make_pipeline(column_transf,KNeighborsRegressor())

# Paso 3. Definir los modelos que vamos a comparar
modelos_candidatos=GridSearchCV(modelo,param_grid=espacio_parametros,
                               scoring="neg_root_mean_squared_error",
                               cv=10,n_jobs=-1)

# Paso 4. Entrenar los modelos candidatos
modelos_candidatos.fit(X,y)

# Imprimimos los resultados:
print("La mejor calibración es:", modelos_candidatos.best_params_)
print("La mejor métrica es:", -modelos_candidatos.best_score_)

La mejor calibración es: {'kneighborsregressor__n_neighbors': 29}
La mejor métrica es: 335.3530108249548


In [39]:
#La mejor calibración es: {'kneighborsregressor__n_neighbors': 29}
#La mejor métrica es: 335.3530108249548

In [40]:
# Cómo entrenamos y hacemos ahora las predicciones
# 1. Vamos a entrenar el Modelo Final empleando TODOS los datos Ojo! 
#    pero con la transformaciones por columnas que aprendamos
#    Vamos a transformar los datos
X_std=column_transf.fit_transform(X)
#   Entrenamos el modelo empleando los datos estandarizados X_std
modelo=KNeighborsRegressor(n_neighbors=29)
modelo.fit(X_std,y)

KNeighborsRegressor(n_neighbors=29)

In [41]:
# Crear un bateador nuevo
X_nuevo=pd.DataFrame({'AtBat':[350], 'Hits':[200], 'HmRun':[5], 
                      'Runs':[25], 'RBI':[18], 'Walks':[5], 'Years':[3],
                      'League':["A"],'Division':["W"]})



X_nuevo

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,League,Division
0,350,200,5,25,18,5,3,A,W


In [42]:
# Antes de hacer la predicción, transformo los datos
# empleando la estrategia aprendida por column_transf
X_nuevo_std=column_transf.transform(X_nuevo)
X_nuevo_std

array([[-0.36484881,  2.04645246, -0.75737255, -1.1668823 , -1.29625265,
        -1.66602953, -0.90120024,  1.        ,  0.        ,  0.        ,
         1.        ]])

In [43]:
modelo.predict(X_nuevo_std)

array([287.10344828])

In [44]:
# Salario de 287.10 +/-335.35

# K vecinos ponderados

In [45]:
# Kernel: Inverso, metrica euclideana, 8 vecinos más cercanos
# Dividimos el conjunto de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test= train_test_split(X,y, 
                                                   test_size=0.20, 
                                                  random_state=1234)
# Creamos las dummies y estandarizamos los datos de entrenamiento
X_dummies_std=column_transf.fit_transform(X_train)
# Definimos el modelo
modelo=KNeighborsRegressor(n_neighbors=8,weights="distance", metric="l2")
modelo.fit(X_dummies_std,y_train)

# Hacemos las predicciones (Transformar los datos de prueba antes)
y_pred=modelo.predict(column_transf.transform(X_test))
y_pred
# El error: Raiz Cuadrada del Error Cuadrática Medio
np.sqrt(((y_test-y_pred)**2).mean())





303.233960208744

# Vamos a calibrar el Algoritmo de K vecinos ponderado

In [46]:

# Optimizar el hiperparámetro. / Calibrar el modelo
# Utilizamos un escalamiento StandarScaler

# paso 1. Definir los posibles valores de los hiperparámetros
espacio_parametros={"kneighborsregressor__n_neighbors":np.arange(3,30)}

# Paso 2. Secuencia para entrenar el modelo
# como existe un preprocesamiento (reescalar los datos)
modelo= make_pipeline(column_transf,KNeighborsRegressor(weights="distance", 
                                                        metric="l2"))

# Paso 3. Definir los modelos que vamos a comparar
modelos_candidatos=GridSearchCV(modelo,param_grid=espacio_parametros,
                               scoring="neg_root_mean_squared_error",
                               cv=10,n_jobs=-1)

# Paso 4. Entrenar los modelos candidatos
modelos_candidatos.fit(X,y)

# Imprimimos los resultados:
print("La mejor calibración es:", modelos_candidatos.best_params_)
print("La mejor métrica es:", -modelos_candidatos.best_score_)

La mejor calibración es: {'kneighborsregressor__n_neighbors': 16}
La mejor métrica es: 331.7267613533486


In [47]:
# Vamos a incluir la métrica dentro del proceso de calibración

In [48]:
# Optimizar el hiperparámetro. / Calibrar el modelo
# Utilizamos un escalamiento StandarScaler

# paso 1. Definir los posibles valores de los hiperparámetros
espacio_parametros={"kneighborsregressor__n_neighbors":np.arange(3,30),
                    "kneighborsregressor__metric":["l1","l2","cosine",
                                                  "manhattan"]
                   }

# Paso 2. Secuencia para entrenar el modelo
# como existe un preprocesamiento (reescalar los datos)
modelo= make_pipeline(column_transf,KNeighborsRegressor(weights="distance"))

# Paso 3. Definir los modelos que vamos a comparar
modelos_candidatos=GridSearchCV(modelo,param_grid=espacio_parametros,
                               scoring="neg_root_mean_squared_error",
                               cv=10,n_jobs=-1)

# Paso 4. Entrenar los modelos candidatos
modelos_candidatos.fit(X,y)

# Imprimimos los resultados:
print("La mejor calibración es:", modelos_candidatos.best_params_)
print("La mejor métrica es:", -modelos_candidatos.best_score_)

La mejor calibración es: {'kneighborsregressor__metric': 'l2', 'kneighborsregressor__n_neighbors': 16}
La mejor métrica es: 331.7267613533486


# Vamos a definir un kernel Epanechnikov

In [49]:
def kernel_Epanechnikov(d):
    weights=3*(1-d**2)/4
    return weights

In [50]:
# Optimizar el hiperparámetro. / Calibrar el modelo
# Utilizamos un escalamiento StandarScaler

# paso 1. Definir los posibles valores de los hiperparámetros
espacio_parametros={"kneighborsregressor__n_neighbors":np.arange(3,30),
                    "kneighborsregressor__metric":["l1","l2","cosine",
                                                  "manhattan"]
                   }

# Paso 2. Secuencia para entrenar el modelo
# como existe un preprocesamiento (reescalar los datos)
modelo= make_pipeline(column_transf,KNeighborsRegressor(weights=kernel_Epanechnikov))

# Paso 3. Definir los modelos que vamos a comparar
modelos_candidatos=GridSearchCV(modelo,param_grid=espacio_parametros,
                               scoring="neg_root_mean_squared_error",
                               cv=10,n_jobs=-1)

# Paso 4. Entrenar los modelos candidatos
modelos_candidatos.fit(X,y)

# Imprimimos los resultados:
print("La mejor calibración es:", modelos_candidatos.best_params_)
print("La mejor métrica es:", -modelos_candidatos.best_score_)

La mejor calibración es: {'kneighborsregressor__metric': 'cosine', 'kneighborsregressor__n_neighbors': 21}
La mejor métrica es: 340.1265386037819


# Bagging
Hasta aquí hemos construido un solo modelo que predice el salario del jugador. Es decir, tenemos un "experto" que en función de las características del jugador nos aproxima su salario. Podríamos ser más eficaces si tuvieramos un número
grande de "expertos" (principio de la inteligencia colectiva).

Para simular los expertos, vamos a entrenar algoritmos con información diferente. Esto lo podemos hacer de varias formas.
1. Cada "experto" lo voy a a entrenar con atributos diferentes
2. No todos conocen la información de TODOS los jugadores. Podemos construir muestras tipo Bootstrap
3. Combinando los dos anteriores. 

In [74]:
datos=pd.read_csv("hitters.csv")

In [75]:
datos

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,League,Division,PutOuts,Assists,Errors,Salary,NewLeague
0,293,66,1,30,29,14,1,293,66,1,30,29,14,A,E,446,33,20,NaN,A
1,315,81,7,24,38,39,14,3449,835,69,321,414,375,N,W,632,43,10,475.0,N
2,479,130,18,66,72,76,3,1624,457,63,224,266,263,A,W,880,82,14,480.0,A
3,496,141,20,65,78,37,11,5628,1575,225,828,838,354,N,E,200,11,3,500.0,N
4,321,87,10,39,42,30,2,396,101,12,48,46,33,N,E,805,40,4,91.5,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,497,127,7,65,48,37,5,2703,806,32,379,311,138,N,E,325,9,3,700.0,N
318,492,136,5,76,50,94,12,5511,1511,39,897,451,875,A,E,313,381,20,875.0,A
319,475,126,3,61,43,52,6,1700,433,7,217,93,146,A,W,37,113,7,385.0,A
320,573,144,9,85,60,78,8,3198,857,97,470,420,332,A,E,1314,131,12,960.0,A


In [76]:
# Eliminamos los NaN en Salario
datos=datos[datos.Salary.isnull()==False]
datos

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,League,Division,PutOuts,Assists,Errors,Salary,NewLeague
1,315,81,7,24,38,39,14,3449,835,69,321,414,375,N,W,632,43,10,475.0,N
2,479,130,18,66,72,76,3,1624,457,63,224,266,263,A,W,880,82,14,480.0,A
3,496,141,20,65,78,37,11,5628,1575,225,828,838,354,N,E,200,11,3,500.0,N
4,321,87,10,39,42,30,2,396,101,12,48,46,33,N,E,805,40,4,91.5,N
5,594,169,4,74,51,35,11,4408,1133,19,501,336,194,A,W,282,421,25,750.0,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,497,127,7,65,48,37,5,2703,806,32,379,311,138,N,E,325,9,3,700.0,N
318,492,136,5,76,50,94,12,5511,1511,39,897,451,875,A,E,313,381,20,875.0,A
319,475,126,3,61,43,52,6,1700,433,7,217,93,146,A,W,37,113,7,385.0,A
320,573,144,9,85,60,78,8,3198,857,97,470,420,332,A,E,1314,131,12,960.0,A


In [77]:
# Vamos a construir las dummies
datos=pd.get_dummies(datos, columns=["League", "Division", "NewLeague"])

In [78]:
datos

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,...,PutOuts,Assists,Errors,Salary,League_A,League_N,Division_E,Division_W,NewLeague_A,NewLeague_N
1,315,81,7,24,38,39,14,3449,835,69,...,632,43,10,475.0,0,1,0,1,0,1
2,479,130,18,66,72,76,3,1624,457,63,...,880,82,14,480.0,1,0,0,1,1,0
3,496,141,20,65,78,37,11,5628,1575,225,...,200,11,3,500.0,0,1,1,0,0,1
4,321,87,10,39,42,30,2,396,101,12,...,805,40,4,91.5,0,1,1,0,0,1
5,594,169,4,74,51,35,11,4408,1133,19,...,282,421,25,750.0,1,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,497,127,7,65,48,37,5,2703,806,32,...,325,9,3,700.0,0,1,1,0,0,1
318,492,136,5,76,50,94,12,5511,1511,39,...,313,381,20,875.0,1,0,1,0,1,0
319,475,126,3,61,43,52,6,1700,433,7,...,37,113,7,385.0,1,0,0,1,1,0
320,573,144,9,85,60,78,8,3198,857,97,...,1314,131,12,960.0,1,0,1,0,1,0


In [79]:
datos.shape

(263, 23)

In [80]:
# Para evitar el problema de la multidimensionalidad, cada "experto" los vamos a entrenar con
# una proporción de los atributos originales, es lo hacemos con el parámetro max_features
from sklearn.ensemble import BaggingRegressor

y=datos.Salary
del datos["Salary"]
X=datos


In [81]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.20, random_state=1234)

In [82]:
X.columns

Index(['AtBat', 'Hits', 'HmRun', 'Runs', 'RBI', 'Walks', 'Years', 'CAtBat',
       'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks', 'PutOuts', 'Assists',
       'Errors', 'League_A', 'League_N', 'Division_E', 'Division_W',
       'NewLeague_A', 'NewLeague_N'],
      dtype='object')

In [83]:
#Preprocesamiento de las columas
column_transf=make_column_transformer(
(StandardScaler(),['AtBat', 'Hits', 'HmRun', 'Runs', 'RBI', 'Walks', 'Years', 'CAtBat',
       'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks', 'PutOuts', 'Assists',
       'Errors']),
remainder="passthrough")

In [84]:
# Definimos el modelo de "expertos"
modelo=BaggingRegressor(KNeighborsRegressor(n_neighbors=5,weights="distance"), 
                        max_features=0.4, n_estimators=20)

In [69]:
# Entrenamos al modelo, pero primero reescalamos los datos para eliminar las unidades de medicion
X_train_std=column_transf.fit_transform(X_train)
# De una vez estandarizamos los de prueba.
X_test_std=column_transf.transform(X_test)
# Entreno el modelo con los datos reescalados
modelo.fit(X_train_std,y_train)

BaggingRegressor(estimator=KNeighborsRegressor(weights='distance'),
                 max_features=0.4, n_estimators=20)

In [70]:
y_pred=modelo.predict(X_test_std)

In [71]:
np.sqrt(((y_pred-y_test)**2).mean())

221.8139391322875

In [72]:
# Calibración del modelo.
# Espacio parametral: número de vecinnos n_neighbors y max_features. 
espacio_parametral= {"baggingregressor__max_features":np.linspace(0.4,1,10),
                     "baggingregressor__max_samples":np.linspace(0.2,1.0,10),
                     "baggingregressor__estimator__n_neighbors":np.arange(3,8)}
    

modelo=make_pipeline(column_transf, BaggingRegressor(KNeighborsRegressor(weights="distance"),
                                                    n_estimators=20))
# Definir los modelos que vamos a comparar
modelos_candidatos=GridSearchCV(modelo,param_grid=espacio_parametral,
                               scoring="neg_root_mean_squared_error",
                               cv=10,n_jobs=-1)

#Entrenar los modelos candidatos
modelos_candidatos.fit(X,y)
# Imprimimos los resultados:
print("La mejor calibración es:", modelos_candidatos.best_params_)
print("La mejor métrica es:", -modelos_candidatos.best_score_)

La mejor calibración es: {'baggingregressor__estimator__n_neighbors': 4, 'baggingregressor__max_features': 0.4, 'baggingregressor__max_samples': 1.0}
La mejor métrica es: 285.1379799867642


In [85]:
# Modelo Final
modelo_final=BaggingRegressor(KNeighborsRegressor(n_neighbors=4, weights="distance"),
                             max_features=0.4, max_samples=1.0)
# Error Cuadrático Medio: 285.14

In [90]:
# Entrenamos al modelo final..
# Utilizanos todos los datos: reescalados
X_std=column_transf.fit_transform(X)
modelo_final.fit(X_std,y)

BaggingRegressor(estimator=KNeighborsRegressor(n_neighbors=4,
                                               weights='distance'),
                 max_features=0.4)

In [95]:
# Este modelo ya lo podemos ocupar para predecir
X_nuevos=pd.DataFrame({'AtBat':[50], 'Hits':[15], 'HmRun':[9], 
                      'Runs':[5], 'RBI':[10], 'Walks':[4], 'Years':[7], 
                      'CAtBat':[400],'CHits':[80], 'CHmRun':[40], 'CRuns':[60], 'CRBI':[45],
                      'CWalks':[20], 'PutOuts':[132], 'Assists':[30],
                      'Errors':[5], 'League_A':[1], 'League_N':[0], 'Division_E':[0],
                      'Division_W':[1],'NewLeague_A':[0], 'NewLeague_N':[1]})
X_nuevos

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,...,CWalks,PutOuts,Assists,Errors,League_A,League_N,Division_E,Division_W,NewLeague_A,NewLeague_N
0,50,15,9,5,10,4,7,400,80,40,...,20,132,30,5,1,0,0,1,0,1


In [97]:
modelo_final.predict(column_transf.transform(X_nuevos))


array([174.26403969])

Estimación: 174.26 +/- 285.14